In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
"""
Computer Vision
CNN - Convolutional Neural Networks
Convolution Layer:
   Number of filers/kernels
   Kernel size
   stride
   padding
Action Function
Pooling Layer: 
   Kernel size
   stride
Batch Normalization
"""

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [4]:
""" Hyperparameters """
batch_size = 64
learning_rate = 0.001
num_epochs = 10

In [6]:
transform = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [7]:
train_dataset = datasets.MNIST(
    './data', train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    './data', train=False, download=True, transform=transform
)

In [8]:
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

In [13]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # convolution layer 1: 6 filters, kernel 5x5, stride=1, padding=0
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=6,
            kernel_size=5,
            stride=1,
            padding=0
        )
        # convolution layer 2: 16 filters, kernel 5x5, stride=1, padding=0
        self.conv2 = nn.Conv2d(
            in_channels=6,
            out_channels=16,
            kernel_size=5,
            stride=1,
            padding=0
        )
        
        # pooling layer 1: kernel size 2x2,  stride=(2,2)
        self.maxpool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )
        
        # fully connected layer 1: takes 5x5x16 -> 120
        self.fc1 = nn.Linear(in_features=5 * 5 * 16, out_features=120)
        # fully connected layer 2: takes 120 inputs, outputs 84 values
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        # fully connected layer 3: takes 84 inputs, outputs 10 values
        self.fc3 = nn.Linear(in_features=84, out_features=10)
        
        # activation function
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # input shape: 32 x 32 x 1
        x = self.conv1(x) # 28 x 28 x 6
        x = self.maxpool(x) # 14 x 14 x 6
        x = self.relu(x) # 14 x 14 x 6
        
        x = self.conv2(x) # 10 x 10 x 16
        x = self.maxpool(x) # 5 x 5 x 16
        x = self.relu(x) # 5 x 5 x 16
        
        x = x.view(-1, 5 * 5 * 16) # flattens the shape into a vector
        x = self.fc1(x) # 120
        x = self.relu(x) # 120
        
        x = self.fc2(x) # 84
        x = self.relu(x) # 84
        x = self.fc3(x) # 10
        return x

In [14]:
model = LeNet5().to(device)

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [16]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")

Epoch [1/10], Loss: 0.2128
Epoch [2/10], Loss: 0.0630
Epoch [3/10], Loss: 0.0424
Epoch [4/10], Loss: 0.0329
Epoch [5/10], Loss: 0.0264
Epoch [6/10], Loss: 0.0229
Epoch [7/10], Loss: 0.0200
Epoch [8/10], Loss: 0.0166
Epoch [9/10], Loss: 0.0142
Epoch [10/10], Loss: 0.0136


In [17]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Accuracy on the test set: {100 * correct / total:.2f}%")

Accuracy on the test set: 98.64%
